In [ ]:
import numpy as np
import scipy.io
import scipy.optimize as opt
import HeatCurrentFunctions as QH

# ─────────────────────────────────────────────────────────────────
# 1. LOAD EXPERIMENTAL DATA
# ─────────────────────────────────────────────────────────────────
data = scipy.io.loadmat('heat_current_data_ind.mat')

x_endpoints = data['x_data'].flatten()   # I_2 axis endpoints
y_endpoints = data['y_data'].flatten()   # I_1 axis endpoints
Q_exp_fW    = data['z_data']             # shape (N_I2, N_I1), in fW

z = Q_exp_fW
x_full = np.linspace(x_endpoints[0], x_endpoints[-1], z.shape[1])  # I_2
y_full = np.linspace(y_endpoints[0], y_endpoints[-1], z.shape[0])  # I_1

CONV_FACTOR = 130.559162011   # fixed: simulation units → fW

# ─────────────────────────────────────────────────────────────────
# 2. FIXED PARAMETERS
# ─────────────────────────────────────────────────────────────────
gamma_local = 0.001
gamma_deph  = 0.0
d           = 0.17
T_h         = 1.2087
T_c         = 0.186
T_local     = 0.186
f_r = 5.6e9

phi_0 = 2.067e-15
h     = 1
w0    = 1.0
E_c = (150e6/f_r) * h

Jm = QH.sm1 + QH.sm2
Jp = QH.sp1 + QH.sp2

# ─────────────────────────────────────────────────────────────────
# 3. FORWARD MODEL
# ─────────────────────────────────────────────────────────────────
def simulate_heat_current_grid(params, I_1_values, I_2_values, use_collective=False):
    """
    Parameters to fit (13 total):
        [0]  gamma_h      — hot bath coupling strength
        [1]  gamma_c      — cold bath coupling strength
        [2]  f_r          — resonator frequency in Hz (sets energy scale)
        [3]  Qf           — resonator quality factor
        [4]  E_j0_1_Hz    — max Josephson energy qubit 1 in Hz
        [5]  E_j0_2_Hz    — max Josephson energy qubit 2 in Hz
        [6]  E_c_Hz       — charging energy in Hz
        [7]  M00, [8] M01 — mutual inductance matrix, row 0
        [9]  M10, [10] M11— mutual inductance matrix, row 1
        [11] O0, [12] O1  — flux offset vector
    """
    (gamma_h, gamma_c, Qf,
     E_j0_1_Hz, E_j0_2_Hz,
     M00, M01, M10, M11,
     O0, O1) = params

    M = np.array([[M00, M01], [M10, M11]])
    O = np.array([[O0], [O1]])

    # Normalize energies by f_r (same convention as notebook)
    E_j0_1 = (E_j0_1_Hz / f_r) * h
    E_j0_2 = (E_j0_2_Hz / f_r) * h
    E_c    = (E_c_Hz    / f_r) * h

    N1, N2 = len(I_1_values), len(I_2_values)
    Q_map  = np.zeros((N2, N1))

    for i, I1 in enumerate(I_1_values):
        for j, I2 in enumerate(I_2_values):
            I_vec  = np.array([[I1], [I2]])
            phi_q  = (M @ I_vec) * phi_0 + O * phi_0
            phi_q1 = phi_q[0, 0]
            phi_q2 = phi_q[1, 0]

            arg1   = np.pi * phi_q1 / phi_0
            arg2   = np.pi * phi_q2 / phi_0
            E_j_q1 = E_j0_1 * abs(np.cos(arg1)) * np.sqrt(1 + d**2 * np.tan(arg1)**2)
            E_j_q2 = E_j0_2 * abs(np.cos(arg2)) * np.sqrt(1 + d**2 * np.tan(arg2)**2)

            f_q1 = (np.sqrt(8 * E_j_q1 * E_c) - E_c) / h
            f_q2 = (np.sqrt(8 * E_j_q2 * E_c) - E_c) / h

            if use_collective:
                rho = QH.rho_ss_termic_collective_sup(
                    f_q1, f_q2,
                    gamma_local, T_local, gamma_deph,
                    T_h, gamma_h, T_c, gamma_c, w0, Qf
                )
                Q = QH.Current_coll(f_q1, f_q2, rho, T_h, gamma_h, w0, Qf, Jm, Jp)
            else:
                rho = QH.rho_ss_termic_indepentend(
                    f_q1, f_q2,
                    gamma_local, T_local, gamma_deph,
                    T_h, gamma_h, T_c, gamma_c, w0, Qf
                )
                Q = QH.Current_ind(f_q1, f_q2, rho, T_h, gamma_h, w0, Qf)

            Q_map[j, i] = Q * CONV_FACTOR

    return Q_map   # shape (N_I2, N_I1), in fW


# ─────────────────────────────────────────────────────────────────
# 4. COST FUNCTION
# ─────────────────────────────────────────────────────────────────
def cost_function(params, I_1_values, I_2_values, Q_exp, use_collective=False):
    try:
        Q_sim     = simulate_heat_current_grid(params, I_1_values, I_2_values, use_collective)
        residuals = (Q_sim - Q_exp).ravel()
        mse       = np.mean(residuals**2)
        print(f"  MSE = {mse:.6e}  |  params = {np.round(params, 6)}")
        return mse
    except Exception as e:
        print(f"  Simulation failed: {e}")
        return 1e30


# ─────────────────────────────────────────────────────────────────
# 5. INITIAL GUESS AND BOUNDS
# ─────────────────────────────────────────────────────────────────
#          gamma_h  gamma_c  f_r       Qf     E_j0_1_Hz  E_j0_2_Hz  E_c_Hz
p0 = np.array([
    0.0075,  0.0075,  5.6e9,  7.18,  55.748e9,  53.725e9,  150e6,
    #  M00      M01     M10     M11     O0           O1
    16.0824, 0.2855, 6.1438, -0.1156, -2.5284/3, -0.8872/3
])

bounds = [
    (0.001,  0.01),    # gamma_h
    (0.001,  0.01),   # gamma_c
    (5e9, 6e9),    # f_r
    (5, 20),    # Qf
    (30e9,  60e9 ),    # E_j0_1_Hz
    (30e9,  60e9 ),    # E_j0_2_Hz
    (100e6,  200e6 ),    # E_c_Hz
    (10.0,  20.0  ),    # M00
    (0.1,   0.3),    # M01
    (3, 8),    # M10
    (-0.2,   0.1),    # M11
    (-1,   -0.1),    # O0
    (-0.3,   -0.1),    # O1
]


# ─────────────────────────────────────────────────────────────────
# 6. TWO-STAGE OPTIMIZATION
# ─────────────────────────────────────────────────────────────────

# --- Stage 1: coarse global search on a downsampled grid ---
# Subsample to every 5th point to keep runtime manageable
step         = 5
I_1_coarse   = y_full[::step]
I_2_coarse   = x_full[::step]
Q_exp_coarse = Q_exp_fW[::step, ::step]

result_global = opt.differential_evolution(
    cost_function,
    bounds  = bounds,
    args    = (I_1_coarse, I_2_coarse, Q_exp_coarse, False),
    maxiter = 200,
    popsize = 12,
    tol     = 1e-5,
    seed    = 42,
    disp    = True,
    workers = -1,    # parallelise across population members
    polish  = False, # we do a dedicated polish step below
)
print("\nStage 1 result:", result_global.x)

# --- Stage 2: local polish on the full grid ---
result_local = opt.minimize(
    cost_function,
    x0     = result_global.x,
    args   = (y_full, x_full, Q_exp_fW, False),
    method = 'Nelder-Mead',
    options = dict(maxiter=2000, xatol=1e-6, fatol=1e-8, disp=True),
)
print("\nFinal fitted parameters:")
param_names = ['gamma_h', 'gamma_c', 'f_r', 'Qf',
               'E_j0_1_Hz', 'E_j0_2_Hz', 'E_c_Hz',
               'M00', 'M01', 'M10', 'M11', 'O0', 'O1']
for name, val in zip(param_names, result_local.x):
    print(f"  {name:12s} = {val:.6g}")

c:\Users\chowdhf4\.conda\envs\TwoQubitHeatValve\Lib\site-packages\scipy\optimize\_differentialevolution.py:518: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,
